## Model Training and Evaluation

In [ ]:
# Imports
import torch
import torchvision


In [19]:
# Hyperparameters
batch_size = 32         # Number of samples in each batch
dim_in = 28*28          # MNIST images are 28x28
hidden_size = 20        # Number of neurons in the hidden layer
dim_out = 10            # MNIST has 10 classes (0-9)

# Defining the model
class DigitRecognitionModel(torch.nn.Module):
    def __init__(self):
            super(DigitRecognitionModel, self).__init__()
            # order doesnt matter here
            self.hidden = torch.nn.Linear(dim_in, hidden_size, bias=True)   # 128 biases (1 per perceptron)
            self.relu = torch.nn.ReLU()
            self.out = torch.nn.Linear(hidden_size, dim_out, bias=True)
            self.softmax = torch.nn.LogSoftmax(dim=-1)  # apply softmax across the output dimension
    
    def forward(self, x):
        # define how to pass x through the network
        x = self.hidden(x)
        x = self.relu(x)
        x = self.out(x)
        x = self.softmax(x)
        return x
    
    def __str__(self):
        return f"DigitRecognitionModel with {sum(p.numel() for p in self.parameters())} parameters"
    


In [20]:
# Defining the input
mnist = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=torchvision.transforms.ToTensor())
mnist_dataloader = torch.utils.data.DataLoader(mnist, batch_size=batch_size)

model = DigitRecognitionModel()
print(model)


DigitRecognitionModel with 15910 parameters


In [21]:
# Defining Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Defining Loss Function
loss_fn = torch.nn.CrossEntropyLoss()


In [22]:
'''
# Model Training Loop (1 epoch)
for x, y in mnist_dataloader:
    x = x.view(x.size(0), -1)  # flatten the input images
    y_hat = model(x)

    loss = loss_fn(y_hat, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print("Total loss for this batch {}".format(loss.item()))
'''

'\n# Model Training Loop (1 epoch)\nfor x, y in mnist_dataloader:\n    x = x.view(x.size(0), -1)  # flatten the input images\n    y_hat = model(x)\n\n    loss = loss_fn(y_hat, y)\n    loss.backward()\n    optimizer.step()\n    optimizer.zero_grad()\n    print("Total loss for this batch {}".format(loss.item()))\n'

In [23]:
# Model Training Loop
for epoch in range(10):
    sum_loss = []
    for x, y in mnist_dataloader:
        x = x.view(x.size(0), -1)  # flatten the input images
        y_hat = model(x)

        loss = loss_fn(y_hat, y)
        loss.backward()
        optimizer.step()
        # zero the gradients before the next step
        optimizer.zero_grad()
        
        # mean loss for this epoch
        sum_loss.append(loss.item())

    mean_epoch = sum(sum_loss) / len(sum_loss)
    print(f"Mean loss for epoch {epoch}: {mean_epoch}")


Mean loss for epoch 0: 0.43839714873333774
Mean loss for epoch 1: 0.26626685366431874
Mean loss for epoch 2: 0.22374821864465871
Mean loss for epoch 3: 0.19435100148022175
Mean loss for epoch 4: 0.17273838531921307
Mean loss for epoch 5: 0.1559686436434587
Mean loss for epoch 6: 0.14278166015992563
Mean loss for epoch 7: 0.1327264528248459
Mean loss for epoch 8: 0.12472156854222219
Mean loss for epoch 9: 0.11838647899565598


In [24]:
# MNIST test set
mnist_test = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=torchvision.transforms.ToTensor())
mnist_testloader = torch.utils.data.DataLoader(mnist_test, batch_size=batch_size)


In [ ]:
# Model Testing
model.eval()

correct = 0
total = 0

with torch.no_grad():   # we dont want to update the weights during testing
    for x, y in mnist_testloader:
        x = x.view(x.size(0), -1)

        output = model(x)
        predictions = output.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total * 100

print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 95.43%


## Interactive Digit Drawing and Prediction

In [26]:
# Imports
import tkinter as tk
from PIL import Image, ImageDraw
import torchvision.transforms as transforms


In [27]:
# Drawing canvas size
WIDTH = 280
HEIGHT = 280

# Black image, like MNIST
image = Image.new("L", (WIDTH, HEIGHT), 0)
draw = ImageDraw.Draw(image)

def draw_digit(event):
    x, y = event.x, event.y

    r = 12

    canvas.create_oval(
        x-r, y-r, x+r, y+r,
        fill="white",
        outline="white"
    )

    draw.ellipse(
        [x-r, y-r, x+r, y+r],
        fill=255
    )

def predict():
    # Resize drawing to MNIST size
    img = image.resize((28, 28))

    # Convert image to tensor
    transform = transforms.ToTensor()
    x = transform(img)

    # Shape: (1, 784)
    x = x.view(1, -1)

    model.eval()

    with torch.no_grad():
        output = model(x)
        prediction = output.argmax(dim=1).item()

    result_label.config(text=f"Prediction: {prediction}")

def clear():
    global image, draw

    canvas.delete("all")

    image = Image.new("L", (WIDTH, HEIGHT), 0)
    draw = ImageDraw.Draw(image)

    result_label.config(text="Prediction: ")


In [29]:
# Window
window = tk.Tk()
window.title("MNIST Digit Recognition")

canvas = tk.Canvas(
    window,
    width=WIDTH,
    height=HEIGHT,
    bg="black"
)

canvas.pack()

# Draw when mouse moves while left button is pressed
canvas.bind("<B1-Motion>", draw_digit)

predict_button = tk.Button(
    window,
    text="Predict",
    command=predict
)

predict_button.pack()

clear_button = tk.Button(
    window,
    text="Clear",
    command=clear
)

clear_button.pack()

result_label = tk.Label(
    window,
    text="Prediction: ",
    font=("Arial", 20)
)

result_label.pack()

window.mainloop()
